# Phase 1 — Data foundation (v3)
Reads the raw CSV files of CICIDS2017 and CIC-IoT2023, cleans them, maps labels to the
8 unified classes, removes exact duplicates across the WHOLE dataset, and creates ONE
stratified 64/16/20 split per dataset (80/20, then 80/20 of the training part; Misrak & Melaku, 2025).
Infiltration and Heartbleed are excluded (Wisanwanichthan & Thammawichai, 2025). Everything is saved unscaled with all columns,
so every later phase (Config 1/2/3, ablation, scenarios) reads the same split.
Nothing in outputs_v2 is touched.

In [ ]:
# Cell 0 — mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Cell 1 — settings, label maps, helpers
import os, glob, json, time
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

ROOT     = '/content/drive/MyDrive/Thesis/'
RAW      = {'cicids2017': ROOT + 'data/cicids2017/', 'ciciot2023': ROOT + 'data/ciciot2023/'}
OUT      = ROOT + 'outputs_v3/data/'
os.makedirs(OUT, exist_ok=True)
SEED     = 42
CHUNK    = 500_000

LABEL_MAP = {
    'cicids2017': {
        'BENIGN': 'BENIGN', 'DoS Hulk': 'DoS', 'DoS GoldenEye': 'DoS', 'DoS slowloris': 'DoS',
        'DoS Slowhttptest': 'DoS', 'Heartbleed': 'DoS', 'DDoS': 'DDoS',
        'FTP-Patator': 'BruteForce', 'SSH-Patator': 'BruteForce',
        'Web Attack - Brute Force': 'WebAttack', 'Web Attack - XSS': 'WebAttack',
        'Web Attack - Sql Injection': 'WebAttack', 'PortScan': 'PortScan', 'Bot': 'Botnet',
    },
    'ciciot2023': {
        'BENIGNTRAFFIC': 'BENIGN', 'BENIGN': 'BENIGN',
        'DDOS-ICMP_FLOOD': 'DDoS', 'DDOS-UDP_FLOOD': 'DDoS', 'DDOS-TCP_FLOOD': 'DDoS',
        'DDOS-PSHACK_FLOOD': 'DDoS', 'DDOS-SYN_FLOOD': 'DDoS', 'DDOS-RSTFINFLOOD': 'DDoS',
        'DDOS-SYNONYMOUSIP_FLOOD': 'DDoS', 'DDOS-ACK_FRAGMENTATION': 'DDoS',
        'DDOS-UDP_FRAGMENTATION': 'DDoS', 'DDOS-ICMP_FRAGMENTATION': 'DDoS',
        'DDOS-SLOWLORIS': 'DDoS', 'DDOS-HTTP_FLOOD': 'DDoS',
        'DOS-UDP_FLOOD': 'DoS', 'DOS-SYN_FLOOD': 'DoS', 'DOS-TCP_FLOOD': 'DoS', 'DOS-HTTP_FLOOD': 'DoS',
        'MIRAI-GREETH_FLOOD': 'Botnet', 'MIRAI-GREIP_FLOOD': 'Botnet', 'MIRAI-UDPPLAIN': 'Botnet',
        'RECON-HOSTDISCOVERY': 'PortScan', 'RECON-OSSCAN': 'PortScan',
        'RECON-PORTSCAN': 'PortScan', 'RECON-PINGSWEEP': 'PortScan', 'VULNERABILITYSCAN': 'PortScan',
        'BRUTEFORCE-WEB': 'BruteForce', 'BRUTEFORCE-XSS': 'BruteForce',
        'BRUTEFORCE-XML': 'BruteForce', 'BRUTEFORCE-SSH': 'BruteForce', 'DICTIONARYBRUTEFORCE': 'BruteForce',
        'DNS_SPOOFING': 'Spoofing', 'MITM-ARPSPOOFING': 'Spoofing',
        'SQLINJECTION': 'WebAttack', 'XSS': 'WebAttack', 'COMMANDINJECTION': 'WebAttack',
        'BACKDOOR_MALWARE': 'WebAttack', 'UPLOADING_ATTACK': 'WebAttack', 'BROWSERHIJACKING': 'WebAttack',
    },
}
CLASSES = sorted(set(LABEL_MAP['cicids2017'].values()) | set(LABEL_MAP['ciciot2023'].values()))

import re
def norm_label(x, ds):
    s = str(x).strip()
    if ds == 'cicids2017':      # the dash in "Web Attack - XSS" is stored with a broken encoding
        s = s.encode('ascii', errors='replace').decode('ascii')
        return re.sub(r'^Web Attack\W+', 'Web Attack - ', s).strip()
    return s.upper()

def log(msg):
    print(time.strftime('%H:%M:%S'), msg, flush=True)

In [ ]:
# Cell 2 — load + clean each dataset (saved as a checkpoint: re-running skips finished work)
# one-off fix: the first run did not map the CICIDS2017 web-attack labels -> move those files aside
rp = OUT + 'cicids2017_cleaning_report.json'
if os.path.exists(rp) and any(k.startswith('Web Attack') for k in json.load(open(rp))['unmapped']):
    for f in ['cleaned', 'train', 'val', 'test']:
        p = OUT + f'cicids2017_{f}.parquet'
        if os.path.exists(p):
            os.rename(p, p.replace('.parquet', '_labelbug.parquet'))
    os.rename(rp, rp.replace('.json', '_labelbug.json'))
    log('moved aside the cicids2017 files built with the label bug')

def load_clean(ds):
    ckpt = OUT + f'{ds}_cleaned.parquet'
    rep_path = OUT + f'{ds}_cleaning_report.json'
    if os.path.exists(ckpt) and os.path.exists(rep_path):
        log(f'{ds}: checkpoint found, skipping load')
        return
    files = sorted(glob.glob(os.path.join(RAW[ds], '*.csv')))
    assert files, f'No CSV files in {RAW[ds]}'
    rep = dict(files=len(files), rows_raw=0, rows_inf_or_nan=0, unmapped={}, dropped_dup_columns=[])
    parts, cols_ref = [], None
    for i, f in enumerate(files, 1):
        for ch in pd.read_csv(f, chunksize=CHUNK, encoding='latin-1', low_memory=False):
            ch.columns = [c.strip() for c in ch.columns]
            dup = [c for c in ch.columns if c.endswith('.1') and c[:-2] in ch.columns]
            if dup:
                rep['dropped_dup_columns'] = sorted(set(rep['dropped_dup_columns']) | set(dup))
                ch = ch.drop(columns=dup)
            lab_col = next(c for c in ch.columns if c.lower() == 'label')
            ch = ch.rename(columns={lab_col: 'label_orig'})
            if cols_ref is None:
                cols_ref = list(ch.columns)
            ch = ch[cols_ref]
            rep['rows_raw'] += len(ch)
            feat = [c for c in cols_ref if c != 'label_orig']
            ch[feat] = ch[feat].apply(pd.to_numeric, errors='coerce').astype('float32')
            ch[feat] = ch[feat].replace([np.inf, -np.inf], np.nan)
            bad = ch[feat].isna().any(axis=1)
            rep['rows_inf_or_nan'] += int(bad.sum())
            ch = ch[~bad]
            uniq = {x: norm_label(x, ds) for x in ch['label_orig'].unique()}
            ch['label_orig'] = ch['label_orig'].map(uniq)
            ch['label'] = ch['label_orig'].map(LABEL_MAP[ds])
            for k, v in ch.loc[ch['label'].isna(), 'label_orig'].value_counts().items():
                rep['unmapped'][k] = rep['unmapped'].get(k, 0) + int(v)
            ch = ch[ch['label'].notna()]
            parts.append(ch)
        log(f'{ds}: {i}/{len(files)} files read, {rep["rows_raw"]:,} rows so far')
    df = pd.concat(parts, ignore_index=True)
    del parts
    df['label_orig'] = df['label_orig'].astype('category')
    df['label'] = df['label'].astype('category')
    # exact duplicates across the whole dataset (features + original label)
    h = pd.util.hash_pandas_object(df.drop(columns=['label']), index=False).to_numpy()
    _, first = np.unique(h, return_index=True)
    keep = np.zeros(len(df), dtype=bool); keep[first] = True
    rep['rows_after_inf_nan_unmapped'] = int(len(df))
    rep['rows_duplicate_removed'] = int((~keep).sum())
    df = df[keep].reset_index(drop=True)
    rep['rows_clean'] = int(len(df))
    df.to_parquet(ckpt, index=False)
    json.dump(rep, open(rep_path, 'w'), indent=2)
    log(f'{ds}: saved {ckpt} ({len(df):,} rows)')

for ds in ['cicids2017', 'ciciot2023']:
    load_clean(ds)

In [ ]:
# Cell 3 — one stratified split per dataset: 80/20, then 80/20 of the training part -> 64/16/20
# (stratified splitting and ratios as in Misrak & Melaku, 2025, Discover Internet of Things).
# Infiltration and Heartbleed are excluded because of their minimal representation
# (Wisanwanichthan & Thammawichai, 2025, Computers).
EXCLUDE = {'Infiltration', 'Heartbleed'}

def make_split(ds):
    done = OUT + f'{ds}_split_v2.json'
    if os.path.exists(done):
        log(f'{ds}: v2 split found, skipping')
        return
    for s in ['train', 'val', 'test']:                     # keep the earlier 70/15/15 files
        p = OUT + f'{ds}_{s}.parquet'
        q = p.replace('.parquet', '_701515.parquet')
        if os.path.exists(p) and not os.path.exists(q):
            os.rename(p, q)
    df = pd.read_parquet(OUT + f'{ds}_cleaned.parquet')
    lab0 = df['label_orig'].astype(str)
    excluded = lab0[lab0.isin(EXCLUDE)].value_counts().to_dict()
    df = df[~lab0.isin(EXCLUDE)].reset_index(drop=True)
    lab = df['label_orig'].astype(str)
    vc = lab.value_counts()
    strat = lab.where(lab.map(vc) >= 20, df['label'].astype(str))   # very rare labels: stratify by class
    idx = np.arange(len(df))
    trv, te = train_test_split(idx, test_size=0.20, random_state=SEED, stratify=strat)
    tr, va = train_test_split(trv, test_size=0.20, random_state=SEED, stratify=strat.iloc[trv])
    rows = {}
    for name, ix in [('train', tr), ('val', va), ('test', te)]:
        part = df.iloc[np.sort(ix)].reset_index(drop=True)
        part['label_orig'] = part['label_orig'].astype(str).astype('category')
        part.to_parquet(OUT + f'{ds}_{name}.parquet', index=False)
        rows[name] = len(ix)
        log(f'{ds}: {name} saved ({len(ix):,} rows)')
    json.dump(dict(scheme='stratified 80/20, then 80/20 of the training part (64/16/20)',
                   reference_split='Misrak & Melaku (2025)', excluded=excluded,
                   reference_exclusion='Wisanwanichthan & Thammawichai (2025)', rows=rows),
              open(done, 'w'), indent=2)

for ds in ['cicids2017', 'ciciot2023']:
    make_split(ds)

In [ ]:
# Cell 4 — data report (for Chapter 3 tables and the weekly update)
report = {}
for ds in ['cicids2017', 'ciciot2023']:
    rep = json.load(open(OUT + f'{ds}_cleaning_report.json'))
    sp = {s: pd.read_parquet(OUT + f'{ds}_{s}.parquet') for s in ['train', 'val', 'test']}
    feat = [c for c in sp['train'].columns if c not in ('label', 'label_orig')]
    tr = sp['train']
    print(f'\n================ {ds} ================')
    print(f"files {rep['files']} | raw rows {rep['rows_raw']:,} | inf/NaN rows {rep['rows_inf_or_nan']:,} | "
          f"duplicates removed {rep['rows_duplicate_removed']:,} | clean rows {rep['rows_clean']:,}")
    print('unmapped labels (dropped):', rep['unmapped'] or 'none')
    print('split:', json.load(open(OUT + f'{ds}_split_v2.json')))
    print('duplicate columns dropped:', rep['dropped_dup_columns'] or 'none')
    print(f'features: {len(feat)}')
    print(f"split rows: train {len(sp['train']):,} | val {len(sp['val']):,} | test {len(sp['test']):,}")
    orig = pd.DataFrame({s: d['label_orig'].astype(str).value_counts() for s, d in sp.items()}).fillna(0).astype(int)
    orig['class'] = orig.index.map(LABEL_MAP[ds])
    print('\nOriginal labels present (rows per split):')
    print(orig.sort_values(['class', 'train'], ascending=[True, False]).to_string())
    absent = sorted(set(LABEL_MAP[ds]) - set(orig.index))
    print('\nLabels in the map but NOT in the data:', absent or 'none')
    cls = pd.DataFrame({s: d['label'].astype(str).value_counts() for s, d in sp.items()}).reindex(CLASSES).fillna(0).astype(int)
    print('\nUnified classes (rows per split):')
    print(cls.to_string())
    neg = {c: int((tr[c] < 0).sum()) for c in feat if (tr[c] < 0).any()}
    const = [c for c in feat if tr[c].nunique() <= 1]
    samp = tr[feat].sample(min(len(tr), 200_000), random_state=SEED)
    same = [(a, b) for i, a in enumerate(feat) for b in feat[i + 1:] if np.allclose(samp[a], samp[b], equal_nan=True)]
    print('\nColumns with negative values (train rows):', neg or 'none')
    print('Constant columns (train):', const or 'none')
    print('Identical column pairs (200k-row sample):', same or 'none')
    report[ds] = dict(rep, n_features=len(feat), split={s: len(d) for s, d in sp.items()},
                      labels_absent=absent, negative_columns=neg, constant_columns=const,
                      identical_pairs=same, classes=cls.to_dict(), original_labels=orig.drop(columns='class').to_dict())
json.dump(report, open(OUT + 'P1_data_report.json', 'w'), indent=2, default=str)
print('\nSaved:', OUT + 'P1_data_report.json')

# dataset version check: names of the first raw CIC-IoT2023 files
print(sorted(os.listdir(RAW['ciciot2023']))[:5], len(os.listdir(RAW['ciciot2023'])))